In [0]:
%run ../00-setup/00_config

In [0]:
import json
from pyspark.sql import Row

MAX_FILE_SIZE = 1000000

landing_file_path = f"{LANDING_REFERENCE_PATH}/sitematrix_raw.json"
raw_text = dbutils.fs.head(landing_file_path, MAX_FILE_SIZE)
data = json.loads(raw_text)

sitematrix = data.get("sitematrix", {})

parsed_rows = []

for key, val in sitematrix.items():
    if key == "count" or not isinstance(val, dict):
        continue
        
    if "site" in val:
        lang_code = val.get("code", "unknown")
        for site in val["site"]:
            parsed_rows.append(Row(
                wiki_code=site.get("dbname"),
                project_name=site.get("sitename"),
                language_code=lang_code,
                site_url=site.get("url")
            ))
            
    elif key == "specials":
        for site in val:
            parsed_rows.append(Row(
                wiki_code=site.get("dbname"),
                project_name=site.get("sitename"),
                language_code=site.get("code", "special"),
                site_url=site.get("url")
            ))

df_parsed = spark.createDataFrame(parsed_rows)



In [0]:
from pyspark.sql.functions import current_timestamp, current_date

df_bronze = (
    df_parsed
    .withColumn("source_file", F.lit(landing_file_path))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("load_date", current_date())
)

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("path", BRONZE_REFERENCE_PATH)
    .saveAsTable(BRONZE_REFERENCE_TABLE)
)

In [0]:
display(spark.table(BRONZE_REFERENCE_TABLE))